In [ ]:
from __future__ import division, print_function, unicode_literals

# import base packages into the namespace for this program
import numpy as np
import pandas as pd
import os
# os.environ['KMP_DUPLICATE_LIB_OK']='True'
import time
import re
import math
from docx import Document

# import packages for data visulatzation 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.offline as py
py.init_notebook_mode(connected=True)
import plotly.graph_objs as go
import plotly.tools as tls
import plotly.figure_factory as ff

# modeling routines from Scikit Learn packages for model develipment
import sklearn.linear_model 
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, confusion_matrix , accuracy_score , classification_report, make_scorer,recall_score
from sklearn.preprocessing import StandardScaler, LabelBinarizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from keras import regularizers

# obtain precision, recall, F1, and support metrics for model evaluation
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score

# specify the set of classifiers being evaluated
from sklearn.ensemble import RandomForestClassifier

import matplotlib as mpl
mpl.rcParams['font.family'] = 'Times New Roman'

import warnings
warnings.filterwarnings("ignore")
# set KMP_DUPLICATE_LIB_OK=TRUE      #  Windows
# set up randam seed
random_seed = 55
import tensorflow as tf
import torch

from sklearn.impute import KNNImputer
import pickle
df_input = pd.read_csv("E:/CNSI_project/demo_and_data_5.0/aim1_dataset_v2.2.csv")

In [ ]:
model_data_DNN = df_input.copy()
model_data_DNN = model_data_DNN.dropna(thresh=(model_data_DNN.shape[1]-1)*0.4, axis=0)
print(model_data_DNN.culture.value_counts())
model_data_DNN

In [ ]:
df_train, df_test = train_test_split(model_data_DNN, test_size=0.2, random_state = random_seed, stratify=model_data_DNN['culture_old'])

In [ ]:
df_train = df_train.drop(columns=['culture_old'], axis=1)
df_test = df_test.drop(columns=['culture_old'], axis=1)
df_test

In [ ]:
# Null fill
columns1 = df_train.columns
index1 = df_train.index
columns2 = df_test.columns
index2 = df_test.index

imputer = KNNImputer(n_neighbors=5, weights='distance')
df_train = imputer.fit_transform(df_train)
df_test = imputer.transform(df_test)

# Save the imputer as a pickle file for later recall.
with open("E:/CNSI_project/demo_and_data_5.0/Processed_data/StandardPickleFiles/imputer_KNNImputer_corrected_aim1_FCN.pickle","wb") as f:
    pickle.dump(imputer, f)

# change to DataFrame
df_train = pd.DataFrame(df_train, columns=columns1, index=index1)
df_train['culture'] = df_train['culture'].astype(np.int16)
df_test = pd.DataFrame(df_test, columns=columns2, index=index2)
df_test['culture'] = df_test['culture'].astype(np.int16)

df_train

In [ ]:
df_test

In [ ]:
df_train.culture.value_counts()

In [ ]:
df_test.culture.value_counts()

In [ ]:
from sklearn.utils import class_weight
def calc_class_weight(total_y):
    my_class_weight = class_weight.compute_class_weight(class_weight="balanced", classes=np.unique(total_y), y=total_y)
    return my_class_weight

In [ ]:
my_class_weight = calc_class_weight(df_train['culture'].values.tolist())
my_class_weight

In [ ]:
my_class_weight = {0:my_class_weight[0], 1:my_class_weight[1]}
my_class_weight

In [ ]:
# my_class_weight = {0:1, 1:5}
# my_class_weight

In [ ]:
# generate X subsets for Modelling 
X_train = df_train.iloc[:,:-1]
X_test = df_test.iloc[:,:-1]
print(X_train.shape, X_test.shape)

In [ ]:
# generate Y subsets for Modelling 
y_train = df_train.iloc[:,-1:]
y_test = df_test.iloc[:,-1:]
y_train.shape, y_test.shape

In [ ]:
# method 1
# MinMaxScaler normalization

# scaler = MinMaxScaler()
# X_train= scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

# print(X_train.shape, X_test.shape)
# X_train[:2]

In [ ]:
# method 2
# StandardScaler normalization
# Data normalisation leads to less volatility during training

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

# Save the scaler as a pickle file for later recall.
with open("E:/CNSI_project/demo_and_data_5.0/Processed_data/StandardPickleFiles/StandardScaler_corrected_aim1_FCN.pickle","wb") as f:
    pickle.dump(scaler, f)

X_test = scaler.transform(X_test)

print(X_train.shape, X_test.shape)
X_train[:2]

In [ ]:
# DNN
# K-fold verification
# develip Neural Network model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, Flatten
n_cols = X_train.shape[1]
# kernel_regularizer=regularizers.l1_l2(l1=0.001,l2=0.001)
def build_model():
    model = Sequential()
    model.add(Dense(200, kernel_regularizer=regularizers.l1(l1=0.001) , activation='relu' , input_shape=(n_cols,)))
    model.add(Dropout(0.5))
    model.add(Dense(100, kernel_regularizer=regularizers.l1(l1=0.001) , activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(50, kernel_regularizer=regularizers.l1(l1=0.001) , activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(2 , activation = "softmax"))
    
    model.compile(optimizer='adam' , loss='sparse_categorical_crossentropy' , metrics = ['accuracy'])
    return model

In [ ]:
num_epochs = 100
k = 4
num_val_samples = X_train.shape[0] // k
all_histories = []
for i in range(k):
    print('processing fold #', i)
    val_data = X_train[i*num_val_samples : (i+1)*num_val_samples]
    val_targets = y_train[i*num_val_samples : (i+1)*num_val_samples]
    
    partial_train_data = np.concatenate([X_train[:i * num_val_samples], X_train[(i+1) * num_val_samples:]], axis=0)
    partial_train_targets = np.concatenate([y_train[:i * num_val_samples], y_train[(i+1) * num_val_samples:]], axis=0)
    
    model = build_model()
    history = model.fit(partial_train_data, partial_train_targets,
                        validation_data = (val_data, val_targets),
                       epochs = num_epochs, batch_size = 64, verbose = 0, class_weight=my_class_weight)
    all_histories.append(history.history)

In [ ]:
average_loss = [np.mean([x['loss'][i] for x in all_histories]) for i in range(num_epochs)]
average_accuracy = [np.mean([x['accuracy'][i] for x in all_histories]) for i in range(num_epochs)]

average_val_loss = [np.mean([x['val_loss'][i] for x in all_histories]) for i in range(num_epochs)]
average_val_accuracy = [np.mean([x['val_accuracy'][i] for x in all_histories]) for i in range(num_epochs)]

In [ ]:
gap = 1
n = int(num_epochs/gap)

loss_gap10 = [np.average(average_loss[gap*i:gap*(i+1)]) for i in range(n)]
acc_gap10 = [np.average(average_accuracy[gap*i:gap*(i+1)]) for i in range(n)]

val_loss_gap10 = [np.average(average_val_loss[gap*i:gap*(i+1)]) for i in range(n)]
val_acc_gap10 = [np.average(average_val_accuracy[gap*i:gap*(i+1)]) for i in range(n)]

In [ ]:
epochs = [each*gap for each in range(1, 1+n)]

plt.figure(figsize = (16,5))
plt.clf() 
plt.plot(epochs, loss_gap10, 'o-', label='Training loss')
plt.plot(epochs, val_loss_gap10, '*-', label='Validation loss')
plt.title('Training and validation loss DNN')
plt.grid(True, linestyle="--", alpha=0.5)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize = (16,5))
# plt.grid(axis="y")
plt.plot(epochs, acc_gap10, 'o-', label='Training acc')
plt.plot(epochs, val_acc_gap10, '*-', label='Validation acc')
plt.title('Training and validation accuracy DNN')
plt.grid(True, linestyle="--", alpha=0.5)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
model = build_model()
model.fit(X_train, y_train, epochs = 38, batch_size = 64, verbose = 1, class_weight=my_class_weight)

In [ ]:
# save model
# model.save()

In [ ]:
# load model
from tensorflow.keras.models import load_model
model = load_model()

# DNN Evaluation

In [ ]:
model.evaluate(X_test, y_test)

In [ ]:
y_predict = model.predict(X_test)
y_predict

In [ ]:
y_test

In [ ]:
# generate confusion matrix
labels1 = ["N", "P"]
plt.figure(figsize = (8, 6))
cm = confusion_matrix(y_test.values,y_predict.argmax(axis=1))
ax = sns.heatmap(cm, square=True, annot=True , cmap="YlGnBu", annot_kws={"fontsize":28}, fmt=".0f")   #square=True, annot=True showing number
# plt.matshow(cm)
# plt.title('Confusion matrix DNN', fontsize=18)
plt.ylabel("True Labels", fontsize=32)
plt.xlabel("Predict Labels", fontsize=32)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=26)

# plt.colorbar()
plt.xticks([i + 0.5 for i in range(len(labels1))], labels1, rotation = 0, fontsize=30)
plt.yticks([i + 0.5 for i in range(len(labels1))], labels1, rotation = 0, fontsize=30)

plt.show()

In [ ]:
cm

In [ ]:
predict_matrix = np.zeros(cm.shape)
for i in range(cm.shape[0]):
    temp = np.sum(cm[i])
    for j in range(cm.shape[1]):
        predict_matrix[i,j] = cm[i,j]/temp
predict_matrix

In [ ]:
plt.figure(figsize = (8, 6))
ax = sns.heatmap(predict_matrix, square=True, annot=True ,cmap="YlGnBu", annot_kws={"fontsize":28}, fmt=".3f", vmin=0, vmax=1)
plt.xticks([i + 0.5 for i in range(len(predict_matrix))], labels1, rotation = 0, fontsize=30)
plt.yticks([i + 0.5 for i in range(len(predict_matrix))], labels1, rotation = 0, fontsize=30)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=26)

# plt.title('Confusion matrix of prediction', fontsize=18)
plt.ylabel("True Labels", fontsize=32)
plt.xlabel("Predict Labels", fontsize=32)

plt.show()

In [ ]:
# calculate evaluation metric using test cohort
accuracy0 = np.round(accuracy_score(y_test.values, 
               y_predict.argmax(axis=1)), decimals = 3)
print('\nDNN Predictive Accuracy: ',accuracy0)

In [ ]:
print('\nSummary of DNN Precision and Recall\n') 
print(classification_report(y_test, y_predict.argmax(axis=1), 
                            target_names = labels1, digits=3))

In [ ]:
from sklearn.metrics import confusion_matrix

y_true = np.array(y_test).flatten()
y_true = y_true.astype(int)
y_pred = y_predict.argmax(axis=1)

cm = confusion_matrix(y_true, y_pred)
TN = cm[0][0]
FP = cm[0][1]
FN = cm[1][0]
TP = cm[1][1]

# Sensitivity, hit rate, recall, or true positive rate
TPR = TP/(TP+FN)
# Specificity or true negative rate
TNR = TN/(TN+FP) 

# The weights for each class
weights = np.bincount(y_true) / len(y_true)

# Weighted average sensitivity and specificity
weighted_avg_sensitivity = np.sum(weights * np.array([TPR, TNR]))
weighted_avg_specificity = np.sum(weights * np.array([TNR, TPR]))

print('Weighted avg sensitivity:', weighted_avg_sensitivity)
print('Weighted avg specificity:', weighted_avg_specificity)

In [ ]:
from sklearn.metrics import confusion_matrix

y_true = np.array(y_test).flatten()
y_true = y_true.astype(int)
y_pred = y_predict.argmax(axis=1)

cm = confusion_matrix(y_true, y_pred)
TN = cm[0][0]
FP = cm[0][1]
FN = cm[1][0]
TP = cm[1][1]

# Sensitivity, hit rate, recall, or true positive rate
sensitivity = TP/(TP+FN)
# Specificity or true negative rate
specificity = TN/(TN+FP) 

print('sensitivity:', sensitivity)
print('specificity:', specificity)

In [ ]:
y_predict

In [ ]:
y_test

In [ ]:
# Calculate AUC score and plot ROC curve
from sklearn.metrics import roc_curve, auc,roc_auc_score

# fpr, tpr, thresholds = roc_curve(label_true.ravel(), label_predict.ravel())
# auc = auc(fpr, tpr)

fpr, tpr, thresholds = roc_curve(y_test, y_predict[:,1])
auc = roc_auc_score(y_test, y_predict[:,1])

plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic Curve')
plt.legend()
plt.show()

In [ ]:
y_test

# AUPRC?C-index and Brier Score

In [ ]:
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score, brier_score_loss
from lifelines.utils import concordance_index

# AUPRC
precision, recall, _ = precision_recall_curve(y_test, y_predict[:,1])
auprc = auc(recall, precision)
print(f'FCN AUPRC: {auprc:.3f}')

baseline = y_test.culture.value_counts()[1] / len(y_test)
print(f'Baseline (positive class ratio): {baseline:.3f}')

plt.figure(figsize=(10, 9.5))
plt.plot(recall, precision, color='blue', label=f'Model (AUPRC = {auprc:.3f})', linewidth=3, alpha=0.7)

plt.axhline(y=baseline, color='red', linestyle='--', label=f'Baseline (AUPRC = {baseline:.3f})', linewidth=2)

plt.xlabel('Recall', fontsize=30, fontname='Times New Roman')
plt.ylabel('Precision', fontsize=30, fontname='Times New Roman')
plt.title('Precision-Recall Curve', fontsize=30, fontname='Times New Roman')

legend = plt.legend(loc='lower left')
for text in legend.get_texts():
    text.set_fontname('Times New Roman')
    text.set_fontsize(25)

plt.grid()

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xticks(fontsize=25, fontname='Times New Roman')
plt.yticks(fontsize=25, fontname='Times New Roman')

plt.show()

In [ ]:
# C-index
# C-index is commonly used for survival analysis, but can also be used for classification tasks to measure the consistency of the model for survival or event ordering.
# For binary classification tasks, you can simply use the concordance_index function in lifelines.utils.
c_index = concordance_index(y_test, y_predict[:,1])
print(f'FCN C-index: {c_index:.3f}')

In [ ]:
# Brier Score 
# Measures the mean square error between the model's predicted probability and the actual outcome.
brier_score = brier_score_loss(y_test, y_predict[:,1])
print(f'FCN Brier Score: {brier_score:.3f}')